--------------------
# Music Recommendation System
--------------------

### Import Libraries

In [711]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix

--------------------
# Load data
--------------------

In [712]:
df = pd.read_csv('data/songs.csv')
df

,artist,song,duration_ms,explicit,year,popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,genre
0,Britney Spears,Oops!...I Did It Again,211160,False,2000,77,0.751,0.834,1,-5.444,0,0.0437,0.3000,0.000018,0.3550,0.894,95.053,pop
1,blink-182,All The Small Things,167066,False,1999,79,0.434,0.897,0,-4.918,1,0.0488,0.0103,0.000000,0.6120,0.684,148.726,"rock, pop"
2,Faith Hill,Breathe,250546,False,1999,66,0.529,0.496,7,-9.007,1,0.0290,0.1730,0.000000,0.2510,0.278,136.859,"pop, country"
3,Bon Jovi,It's My Life,224493,False,2000,78,0.551,0.913,0,-4.063,0,0.0466,0.0263,0.000013,0.3470,0.544,119.992,"rock, metal"
4,*NSYNC,Bye Bye Bye,200560,False,2000,65,0.614,0.928,8,-4.806,0,0.0516,0.0408,0.001040,0.0845,0.879,172.656,pop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Jonas Brothers,Sucker,181026,False,2019,79,0.842,0.734,1,-5.065,0,0.0588,0.0427,0.000000,0.1060,0.952,137.958,pop
1996,Taylor Swift,Cruel Summer,178426,False,2019,78,0.552,0.702,9,-5.707,1,0.1570,0.1170,0.000021,0.1050,0.564,169.994,pop
1997,Blanco Brown,The Git Up,200593,False,2019,69,0.847,0.678,9,-8.635,1,0.1090,0.0669,0.000000,0.2740,0.811,97.984,"hip hop, country"
1998,Sam Smith,Dancing With A Stranger (with Normani),171029,False,2019,75,0.741,0.520,8,-7.513,1,0.0656,0.4500,0.000002,0.2220,0.347,102.998,pop


In [713]:
df = df.drop_duplicates(subset=['artist', 'song']).reset_index(drop=True)

In [714]:
df.isna().sum()

artist              0
song                0
duration_ms         0
explicit            0
year                0
popularity          0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
genre               0
dtype: int64

In [715]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1926 entries, 0 to 1925
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   artist            1926 non-null   str    
 1   song              1926 non-null   str    
 2   duration_ms       1926 non-null   int64  
 3   explicit          1926 non-null   bool   
 4   year              1926 non-null   int64  
 5   popularity        1926 non-null   int64  
 6   danceability      1926 non-null   float64
 7   energy            1926 non-null   float64
 8   key               1926 non-null   int64  
 9   loudness          1926 non-null   float64
 10  mode              1926 non-null   int64  
 11  speechiness       1926 non-null   float64
 12  acousticness      1926 non-null   float64
 13  instrumentalness  1926 non-null   float64
 14  liveness          1926 non-null   float64
 15  valence           1926 non-null   float64
 16  tempo             1926 non-null   float64
 17  genre 

 --------------------
# Audio features
 --------------------

In [716]:
features = [
    'danceability',
    'energy',
    'loudness',
    'tempo',
    'valence',
    'acousticness',
    'instrumentalness',
    'liveness',
    'speechiness',
    'mode',
    'key'
]

In [717]:
scaler = StandardScaler()
audio_features = scaler.fit_transform(df[features])

In [718]:
audio_features[:,3] *= 0.7
audio_features[:,8] *= 0.5

 --------------------
# Genre multi label
 --------------------

In [719]:
genre_lists = (df['genre'].str.lower().str.split(',').apply(lambda x: [g.strip() for g in x if g.strip()]))

mlb = MultiLabelBinarizer()

genre_features = mlb.fit_transform(genre_lists)

In [720]:
df['genre'].value_counts()

genre
pop                                      411
hip hop, pop                             265
hip hop, pop, R&B                        234
pop, Dance/Electronic                    213
pop, R&B                                 170
hip hop                                  120
hip hop, pop, Dance/Electronic            75
rock                                      57
Dance/Electronic                          41
rock, pop                                 39
rock, metal                               36
pop, latin                                28
pop, rock                                 26
set()                                     22
hip hop, Dance/Electronic                 15
latin                                     15
pop, rock, metal                          14
hip hop, pop, latin                       14
R&B                                       13
pop, rock, Dance/Electronic               12
metal                                      9
hip hop, pop, rock                         9
coun

--------------------
# Combine
--------------------

In [721]:
final_features = hstack([
    csr_matrix(audio_features * 0.7),
    csr_matrix(genre_features * 0.3),
]).tocsr()

In [722]:
model = NearestNeighbors(
    n_neighbors=10,
    metric='cosine'
)

model.fit(final_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


--------------------
# Model Training
--------------------


In [723]:
model = NearestNeighbors(n_neighbors=10, metric='cosine')

model.fit(final_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


-------------
### Funcion to get recommendations based on song name
-------------

In [724]:
def recommendation(song_name):

    result = df[df['song'].str.contains(song_name, case=False, na=False)]

    if result.empty:
        print("Song not found")
        return

    song_index = result.index[0]

    target_genre = set(genre_lists[song_index])

    candidates = df[genre_lists.apply(lambda x: len(target_genre.intersection(set(x))) > 0)]

    candidate_indexes = candidates.index

    distances, indices = model.kneighbors(
        final_features[song_index],
        n_neighbors=50
    )


    print("Recommendations for:", df.iloc[song_index]['song'])

    print("-----------------------------")


    count = 0

    for distance, index in zip(
        distances[0][1:],
        indices[0][1:]
    ):

        if index in candidate_indexes:

            print(
                df.iloc[index]['artist'],
                "-",
                df.iloc[index]['song'],
                "| distance:",
                round(distance,3)
            )

            count += 1

        if count == 9:
            break

------------
### Recommendations
------------

In [726]:
recommendation('Shape of You')

Recommendations for: Shape of You
-----------------------------
D12 - My Band | distance: 0.092
Ne-Yo - Because Of You | distance: 0.1
Ashanti - Rock Wit U (Awww Baby) | distance: 0.119
Britney Spears - I'm a Slave 4 U | distance: 0.128
Destiny's Child - Independent Women, Pt. 1 | distance: 0.161
Britney Spears - Break the Ice | distance: 0.175
TiÃ«sto - Jackie Chan | distance: 0.183
Akon - Smack That | distance: 0.204
Destiny's Child - Say My Name | distance: 0.217


In [727]:
recommendation('Cheap Thrills')

Recommendations for: Cheap Thrills (feat. Sean Paul)
-----------------------------
G-Unit - Wanna Get To Know You | distance: 0.145
Shawn Mendes - Treat You Better | distance: 0.148
Selena Gomez - Come & Get It | distance: 0.175
Selena Gomez - The Heart Wants What It Wants | distance: 0.176
Busta Rhymes - Pass The Courvoisier Part II (feat. P. Diddy & Pharrell) - Remix | distance: 0.183
JoJo - Too Little Too Late | distance: 0.184
Bruno Mars - Finesse - Remix; feat. Cardi B | distance: 0.184
50 Cent - P.I.M.P. | distance: 0.195
Maroon 5 - This Love | distance: 0.21


In [728]:
recommendation('Shape of My Heart')

Recommendations for: Shape of My Heart
-----------------------------
Vanessa Carlton - A Thousand Miles | distance: 0.096
The Fray - How to Save a Life | distance: 0.16
CÃ©line Dion - That's the Way It Is | distance: 0.171
Jason Derulo - Ridin' Solo | distance: 0.181
Shontelle - Impossible | distance: 0.192
The Script - Hall of Fame (feat. will.i.am) | distance: 0.215
Kelly Clarkson - Already Gone | distance: 0.23
Bruno Mars - Marry You | distance: 0.23
David Guetta - Hey Mama (feat. Nicki Minaj, Bebe Rexha & Afrojack) | distance: 0.241


In [729]:
recommendation('Let Her Go')

Recommendations for: Let Her Go
-----------------------------
Robbie Williams - Eternity | distance: 0.049
Shayne Ward - No Promises | distance: 0.069
The Pussycat Dolls - Stickwitu | distance: 0.074
Ronan Keating - If Tomorrow Never Comes | distance: 0.074
Alicia Keys - Empire State of Mind (Part II) Broken Down | distance: 0.081
Birdy - Let It All Go | distance: 0.097
Westlife - The Rose | distance: 0.099
Westlife - You Raise Me Up | distance: 0.099
Christina Perri - Jar of Hearts | distance: 0.101


In [730]:
recommendation('Side to Side')

Recommendations for: Side To Side
-----------------------------
Katy Perry - Roar | distance: 0.232
Christina Aguilera - Candyman | distance: 0.239
Huey - Pop, Lock & Drop It - Video Edit | distance: 0.249
JLS - Everybody in Love | distance: 0.263
The Veronicas - Untouched | distance: 0.265
BeyoncÃ© - Irreplaceable | distance: 0.283
Bobby V. - Slow Down - 12" Version | distance: 0.285
Lil Wayne - A Milli | distance: 0.285
T.I. - Whatever You Like | distance: 0.285
